<a href="https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/intro-prompt-engineering-lesson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What This Notebook Teaches You**
- Understand **: Environment Setup**
- Understand **: How LLMs Actually Generate Text**
- Understand **: Tokenization — How Text Becomes Numbers**
- Understand **: The Embedding Space — Where Meaning Lives**


# Prompt Engineering: Foundations — Understanding LLMs from First Principles

## What This Notebook Teaches You

Most prompt engineering tutorials show you **what to type**. This one shows you **why it works**.

By the end of this notebook, you will understand:

1. **How LLMs actually generate text** — the autoregressive loop, one token at a time
2. **Tokenization** — how text becomes numbers (and why "Hello" ≠ "hello" to a model)
3. **Embeddings** — how tokens become vectors that encode meaning
4. **Attention** — how the model decides what context matters for each prediction
5. **Generation algorithms** — greedy, sampling, top-k, top-p, temperature (with math)
6. **Chat templates** — the hidden formatting that separates system/user/assistant messages
7. **Quantization** — why we can shrink a 14B parameter model to fit a free GPU

Every concept includes **runnable code** so you can inspect the internals yourself. This foundation will make every subsequent prompt engineering technique intuitive rather than magical.

---

> **Prerequisites**: Basic Python. No ML experience needed.  
> **Runtime**: Google Colab with T4 GPU (free tier works).  
> **Time**: ~45-60 minutes to work through everything.

## Part 0: Environment Setup

We load **Qwen3-8B**, a 14.8-billion-parameter open-source model, using 4-bit quantization to fit it on a free Colab T4 GPU (16GB VRAM). We cache the model weights on Google Drive so subsequent loads are instant.

> **Why this model?** Qwen3-8B (released 2025) is one of the most capable open-source models that fits on consumer hardware. It supports 100+ languages, has a 128K token context window, and includes a native "thinking" mode for chain-of-thought reasoning.

In [1]:
# --- Google Colab Setup ---
# Install dependencies (run once)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import numpy as np
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"Vocabulary size: {len(tokenizer):,} tokens")

Mounted at /content/drive


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen3-8B
Parameters: 4,717,851,648
GPU memory used: 6.0 GB
Vocabulary size: 151,669 tokens


---

## Part 1: How LLMs Actually Generate Text

### The Core Idea: Next-Token Prediction

An LLM does exactly **one thing**: given a sequence of tokens, predict the probability distribution over what the **next token** should be.

That's it. Everything else — conversations, code generation, reasoning, poetry — emerges from doing this one thing really well, billions of times during training.

```
Input:  "The capital of France is"
         ↓ model processes all tokens ↓
Output: probability distribution over 150,000+ possible next tokens
         "Paris" → 0.82
         "the"   → 0.03
         "a"     → 0.02
         "Lyon"  → 0.01
         ...every other token → tiny probabilities
```

### The Autoregressive Loop

"Autoregressive" means each prediction feeds back as input for the next. The model generates text **one token at a time**:

1. Process input tokens → predict next token
2. Append predicted token to the sequence
3. Process the whole sequence again → predict next token
4. Repeat until a stop token or max length

Let's watch this happen in real time:

In [2]:
# === THE AUTOREGRESSIVE LOOP: Watch the model generate one token at a time ===

prompt = "The three primary colors are"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

print(f"Starting prompt: '{prompt}'")
print(f"Prompt token IDs: {input_ids[0].tolist()}")
print(f"Number of input tokens: {input_ids.shape[1]}")
print("\n--- Token-by-token generation ---\n")

generated_ids = input_ids.clone()

for step in range(15):  # Generate 15 tokens
    # Step 1: Forward pass — model processes ALL tokens so far
    with torch.no_grad():
        outputs = model(generated_ids)

    # Step 2: Get logits for the LAST position only
    # outputs.logits shape: [batch=1, sequence_length, vocab_size]
    next_token_logits = outputs.logits[0, -1, :]  # shape: [vocab_size]

    # Step 3: Apply softmax to convert logits → probabilities
    probabilities = torch.softmax(next_token_logits, dim=-1)

    # Step 4: Pick the highest-probability token (greedy decoding)
    next_token_id = torch.argmax(probabilities).unsqueeze(0).unsqueeze(0)
    next_token_text = tokenizer.decode(next_token_id[0])

    # Show what happened at this step
    top_prob = probabilities[next_token_id[0, 0]].item()
    print(f"Step {step+1:2d}: predicted '{next_token_text}' (probability: {top_prob:.4f})")

    # Step 5: Append the new token and repeat
    generated_ids = torch.cat([generated_ids, next_token_id], dim=1)

    # Stop if we hit end-of-sequence
    if next_token_id[0, 0].item() == tokenizer.eos_token_id:
        break

# Show the full generated text
full_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(f"\n--- Complete output ---\n{full_text}")

Starting prompt: 'The three primary colors are'
Prompt token IDs: [785, 2326, 6028, 7987, 525]
Number of input tokens: 5

--- Token-by-token generation ---

Step  1: predicted ' red' (probability: 0.6953)
Step  2: predicted ',' (probability: 1.0000)
Step  3: predicted ' yellow' (probability: 0.7344)
Step  4: predicted ',' (probability: 0.9453)
Step  5: predicted ' and' (probability: 1.0000)
Step  6: predicted ' blue' (probability: 0.9961)
Step  7: predicted '.' (probability: 0.8828)
Step  8: predicted ' When' (probability: 0.3184)
Step  9: predicted ' these' (probability: 0.5508)
Step 10: predicted ' three' (probability: 0.4727)
Step 11: predicted ' colors' (probability: 0.9805)
Step 12: predicted ' are' (probability: 0.9961)
Step 13: predicted ' mixed' (probability: 0.9805)
Step 14: predicted ' in' (probability: 0.9023)
Step 15: predicted ' equal' (probability: 0.9453)

--- Complete output ---
The three primary colors are red, yellow, and blue. When these three colors are mixed in equ

### What Just Happened

Look at the output above carefully:

1. **Each step processed the ENTIRE sequence** — not just the last token. The model sees all prior context every time.
2. **The model outputs probabilities over ALL ~150,000 tokens** in its vocabulary, but we only pick one.
3. **High probability ≠ certain** — even "obvious" continuations like "red" after "primary colors are" might only be 60-80% probable.

This is the fundamental loop that powers every LLM interaction. When you "chat" with an AI, it's running this loop hundreds of times to produce each response.

### Inspecting the Probability Distribution

Let's look at what OTHER tokens the model considered at a specific step:

In [4]:
# === INSPECT THE TOP CANDIDATES at each position ===

prompt = "Machine learning is NOT a subset of"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(input_ids)

# Get probabilities for the next token
logits = outputs.logits[0, -1, :]
probabilities = torch.softmax(logits, dim=-1)

# Show top 15 candidates
top_k_probs, top_k_ids = torch.topk(probabilities, k=15)

print(f"Prompt: '{prompt}'")
print(f"\nTop 15 predicted next tokens:")
print(f"{'Rank':<6} {'Token':<20} {'Probability':<12} {'Cumulative':<12}")
print("-" * 50)

cumulative = 0.0
for rank, (prob, token_id) in enumerate(zip(top_k_probs, top_k_ids), 1):
    token_text = tokenizer.decode([token_id.item()])
    cumulative += prob.item()
    print(f"{rank:<6} '{token_text}'{'':>{18-len(token_text)}} {prob.item():<12.6f} {cumulative:<12.4f}")

print(f"\nRemaining {len(tokenizer) - 15:,} tokens share the other {1-cumulative:.4f} probability")

Prompt: 'Machine learning is NOT a subset of'

Top 15 predicted next tokens:
Rank   Token                Probability  Cumulative  
--------------------------------------------------
1      ' statistics'        0.339844     0.3398      
2      ' AI'                0.263672     0.6035      
3      ' artificial'        0.085938     0.6895      
4      ' data'              0.075684     0.7651      
5      ' computer'          0.031494     0.7966      
6      ' Artificial'        0.013184     0.8098      
7      ' deep'              0.013184     0.8230      
8      ' statistical'       0.010254     0.8333      
9      ' machine'           0.007996     0.8412      
10     ' the'               0.007996     0.8492      
11     ' mathematics'       0.006226     0.8555      
12     ' traditional'       0.005493     0.8610      
13     ' probability'       0.004852     0.8658      
14     ' pattern'           0.004852     0.8707      
15     ' Statistics'        0.004852     0.8755      

Remaini

### Key Insight: Why Prompts Matter

Notice how the probability distribution depends entirely on the input sequence. **Changing the prompt changes which tokens are likely.** This is the entire basis of prompt engineering — you're sculpting the probability landscape that the model samples from.

> **Exercise**: Modify the prompt above to "Machine learning is NOT a subset of" and re-run. How do the top predictions change? The word "NOT" fundamentally shifts the distribution.

---

## Part 2: Tokenization — How Text Becomes Numbers

### Why Tokenization Matters

LLMs don't see text. They see **sequences of integers** (token IDs). Before any processing happens, your prompt goes through a **tokenizer** that:

1. Splits text into subword pieces (tokens)
2. Maps each piece to an integer ID from a fixed vocabulary
3. Feeds those integers to the model

Understanding tokenization explains many "weird" LLM behaviors:
- Why models struggle with character counting ("How many r's in strawberry?")
- Why some languages use more tokens (and thus more compute) than others
- Why code formatting affects model output
- Why there's a maximum context length

In [2]:
# === BASIC TOKENIZATION: Text ↔ Token IDs ===

text = "Hello, world! This is prompt engineering."

# Encode: text → token IDs
token_ids = tokenizer.encode(text)
print(f"Text: '{text}'")
print(f"Token IDs: {token_ids}")
print(f"Number of tokens: {len(token_ids)}")

# Decode: token IDs → text
decoded = tokenizer.decode(token_ids)
print(f"Decoded back: '{decoded}'")

# Show each token individually
print(f"\nToken-by-token breakdown:")
print(f"{'ID':<10} {'Token text':<20} {'Bytes'}")
print("-" * 45)
for tid in token_ids:
    token_text = tokenizer.decode([tid])
    token_bytes = tokenizer.convert_ids_to_tokens([tid])[0]
    print(f"{tid:<10} '{token_text}'{'':>{17-len(token_text)}} {token_bytes}")

Text: 'Hello, world! This is prompt engineering.'
Token IDs: [9707, 11, 1879, 0, 1096, 374, 9934, 14667, 13]
Number of tokens: 9
Decoded back: 'Hello, world! This is prompt engineering.'

Token-by-token breakdown:
ID         Token text           Bytes
---------------------------------------------
9707       'Hello'             Hello
11         ','                 ,
1879       ' world'            Ġworld
0          '!'                 !
1096       ' This'             ĠThis
374        ' is'               Ġis
9934       ' prompt'           Ġprompt
14667      ' engineering'      Ġengineering
13         '.'                 .


### Subword Tokenization (BPE)

Modern LLMs use **Byte-Pair Encoding (BPE)** or similar algorithms. The key idea: common words get their own token, rare words are split into pieces.

- "the" → 1 token (very common word)
- "unhappiness" → might be "un" + "happiness" or "un" + "happ" + "iness"
- "supercalifragilistic" → many small pieces

This is a **compression trade-off**: a vocabulary of ~150K tokens balances between having a token for every word (too many) and spelling out every character (too slow).

In [3]:
# === SUBWORD TOKENIZATION IN ACTION ===

test_words = [
    "the",
    "hello",
    "Hello",      # Capitalization matters!
    " Hello",     # Leading space matters!
    "unhappiness",
    "artificial",
    "AI",
    "transformers",
    "supercalifragilistic",
    "antidisestablishmentarianism",
    "12345",
    "3.14159",
    "def fibonacci(n):",
    "🎉",
    "こんにちは",   # Japanese
    "مرحبا",       # Arabic
]

print(f"Vocabulary size: {len(tokenizer):,} tokens\n")
print(f"{'Text':<35} {'Tokens':<6} {'Token pieces'}")
print("-" * 80)

for word in test_words:
    ids = tokenizer.encode(word, add_special_tokens=False)
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"'{word}'{'':>{33-len(word)}} {len(ids):<6} {pieces}")

Vocabulary size: 151,669 tokens

Text                                Tokens Token pieces
--------------------------------------------------------------------------------
'the'                               1      ['the']
'hello'                             1      ['hello']
'Hello'                             1      ['Hello']
' Hello'                            1      [' Hello']
'unhappiness'                       3      ['un', 'h', 'appiness']
'artificial'                        2      ['art', 'ificial']
'AI'                                1      ['AI']
'transformers'                      2      ['transform', 'ers']
'supercalifragilistic'              7      ['sup', 'erc', 'al', 'if', 'rag', 'il', 'istic']
'antidisestablishmentarianism'      6      ['ant', 'idis', 'establish', 'ment', 'arian', 'ism']
'12345'                             5      ['1', '2', '3', '4', '5']
'3.14159'                           7      ['3', '.', '1', '4', '1', '5', '9']
'def fibonacci(n):'                 4   

### Critical Insight: Why "Hello" ≠ "hello" ≠ " Hello"

Look at the output above. The model sees **completely different token IDs** for:
- `"Hello"` (start of text)
- `"hello"` (lowercase)
- `" Hello"` (with leading space)

This means the model treats these as literally different inputs at the most fundamental level. This is why:
- **Capitalization affects output quality** — the model learned different patterns for "Write" vs "write"
- **Spacing matters** — tokens with and without leading spaces are different vocabulary entries
- **The tokenizer is the first transformation** — understanding it helps you predict what the model "sees"

### The Token Economy: Why Token Count Matters

Every token costs compute. The model's context window (128K tokens for Qwen3) is measured in tokens, not characters. Different content types use tokens very differently:

In [4]:
# === TOKEN ECONOMY: How different content types use tokens ===

examples = {
    "Simple English": "The cat sat on the mat and looked at the bird outside the window.",
    "Technical prose": "Gradient descent iteratively optimizes the loss function by computing partial derivatives.",
    "Python code": "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)",
    "JSON data": '{"name": "Alice", "age": 30, "city": "London", "hobbies": ["reading", "coding"]}',
    "Chinese text": "人工智能是计算机科学的一个分支，旨在创建能够执行通常需要人类智能的任务的系统。",
    "Numbers": "1234567890 9876543210 1111111111 2222222222 3333333333",
    "Repeated text": "the " * 15,
}

print(f"{'Content Type':<20} {'Characters':<12} {'Tokens':<10} {'Chars/Token':<12}")
print("-" * 55)

for name, text in examples.items():
    tokens = tokenizer.encode(text, add_special_tokens=False)
    ratio = len(text) / len(tokens)
    print(f"{name:<20} {len(text):<12} {len(tokens):<10} {ratio:<12.1f}")

print("\n💡 Key insight: The model 'sees' Chinese text as MORE tokens per character,")
print("   meaning it uses more of the context window for the same semantic content.")

Content Type         Characters   Tokens     Chars/Token 
-------------------------------------------------------
Simple English       65           15         4.3         
Technical prose      90           14         6.4         
Python code          92           28         3.3         
JSON data            80           29         2.8         
Chinese text         39           19         2.1         
Numbers              54           54         1.0         
Repeated text        60           16         3.8         

💡 Key insight: The model 'sees' Chinese text as MORE tokens per character,
   meaning it uses more of the context window for the same semantic content.


In [ ]:
# === SPECIAL TOKENS: The hidden control characters ===

print("=== Special Tokens in the Vocabulary ===\n")

# Show important special tokens
special_tokens = {
    "BOS (beginning of sequence)": tokenizer.bos_token,
    "EOS (end of sequence)": tokenizer.eos_token,
    "PAD (padding)": tokenizer.pad_token,
    "UNK (unknown)": tokenizer.unk_token,
}

for name, token in special_tokens.items():
    if token:
        tid = tokenizer.convert_tokens_to_ids(token)
        print(f"{name}: '{token}' → ID {tid}")
    else:
        print(f"{name}: (not defined)")

# Show all special tokens the model uses
print(f"\n=== All Special Tokens ===")
for token, tid in sorted(tokenizer.get_added_vocab().items(), key=lambda x: x[1]):
    if tid < 151700:  # Skip very high IDs
        continue
    print(f"  ID {tid}: '{token}'")

> **Exercise**: Try tokenizing a paragraph from your own domain (legal text, medical notes, code). How does the token count compare to plain English? This directly affects how much context you can fit in a single prompt.

---

## Part 3: The Embedding Space — Where Meaning Lives

### From Token IDs to Vectors

Token IDs are just arbitrary integers. The model needs something richer — a representation where **similar concepts are near each other**. This is what the **embedding layer** provides.

Each token ID maps to a high-dimensional vector (typically 1024-8192 dimensions). These vectors are **learned during training** and encode semantic relationships:

```
"king"  → [0.21, -0.45, 0.87, ..., 0.33]   (5120 dimensions)
"queen" → [0.19, -0.41, 0.85, ..., 0.31]   (similar vector! same semantic space)
"apple" → [-0.72, 0.15, -0.23, ..., 0.91]  (very different vector)
```

In [5]:
# === INSPECTING THE EMBEDDING MATRIX ===

# The embedding layer is the first layer of the model
embedding_layer = model.model.embed_tokens

print(f"Embedding matrix shape: {embedding_layer.weight.shape}")
print(f"  → {embedding_layer.weight.shape[0]:,} tokens × {embedding_layer.weight.shape[1]} dimensions")
print(f"  → Each token is represented by a {embedding_layer.weight.shape[1]}-dimensional vector")
print(f"  → Total embedding parameters: {embedding_layer.weight.numel():,}")

# Look up embeddings for specific tokens
words = ["king", "queen", "man", "woman", "apple", "orange", "code", "program"]
print(f"\n{'Token':<12} {'Token ID':<10} {'Embedding (first 8 dims)'}")
print("-" * 65)

for word in words:
    token_ids = tokenizer.encode(word, add_special_tokens=False)
    if len(token_ids) == 1:  # Only show single-token words
        emb = embedding_layer.weight[token_ids[0]].detach().float().cpu().numpy()
        emb_str = ", ".join(f"{x:.3f}" for x in emb[:8])
        print(f"{word:<12} {token_ids[0]:<10} [{emb_str}, ...]")

Embedding matrix shape: torch.Size([151936, 4096])
  → 151,936 tokens × 4096 dimensions
  → Each token is represented by a 4096-dimensional vector
  → Total embedding parameters: 622,329,856

Token        Token ID   Embedding (first 8 dims)
-----------------------------------------------------------------
king         10566      [-0.006, -0.013, 0.049, -0.020, 0.023, -0.050, 0.046, 0.016, ...]
queen        93114      [-0.048, 0.028, -0.018, -0.015, 0.021, -0.028, 0.011, 0.010, ...]
man          1515       [-0.001, 0.007, 0.044, -0.015, 0.024, -0.068, -0.030, -0.028, ...]
woman        22028      [-0.020, 0.010, 0.034, -0.011, 0.021, -0.035, -0.049, -0.038, ...]
apple        22377      [-0.042, 0.011, 0.002, -0.001, 0.007, 0.005, -0.029, -0.012, ...]
orange       34164      [-0.006, 0.039, -0.013, 0.021, 0.016, 0.005, 0.004, -0.016, ...]
code         1851       [0.049, 0.016, 0.012, -0.040, -0.020, -0.048, 0.005, 0.017, ...]
program      14906      [0.009, 0.002, -0.021, -0.001, -0.019, 

In [6]:
# === SEMANTIC SIMILARITY IN EMBEDDING SPACE ===

from numpy.linalg import norm

def get_token_embedding(word):
    """Get the embedding vector for a single-token word."""
    ids = tokenizer.encode(word, add_special_tokens=False)
    emb = embedding_layer.weight[ids[0]].detach().float().cpu().numpy()
    return emb

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (norm(a) * norm(b))

# Compare semantic similarities
pairs = [
    ("king", "queen"),      # Both royalty — should be similar
    ("king", "man"),        # Both male — somewhat similar
    ("king", "apple"),      # Unrelated — should be dissimilar
    ("code", "program"),    # Related concepts — should be similar
    ("happy", "sad"),       # Opposites — interestingly, often similar (same semantic field!)
    ("dog", "cat"),         # Same category — similar
    ("dog", "algorithm"),   # Unrelated — dissimilar
]

print(f"{'Pair':<25} {'Cosine Similarity':<20} {'Interpretation'}")
print("-" * 70)

for w1, w2 in pairs:
    try:
        emb1 = get_token_embedding(w1)
        emb2 = get_token_embedding(w2)
        sim = cosine_similarity(emb1, emb2)
        if sim > 0.5:
            interp = "Very similar"
        elif sim > 0.3:
            interp = "Somewhat similar"
        elif sim > 0.1:
            interp = "Slightly related"
        else:
            interp = "Unrelated"
        print(f"({w1}, {w2}){'':>{22-len(w1)-len(w2)}} {sim:<20.4f} {interp}")
    except Exception as e:
        print(f"({w1}, {w2}){'':>{22-len(w1)-len(w2)}} (multi-token, skipping)")

Pair                      Cosine Similarity    Interpretation
----------------------------------------------------------------------
(king, queen)              0.0743               Unrelated
(king, man)                0.0517               Unrelated
(king, apple)              0.0131               Unrelated
(code, program)            0.0913               Unrelated
(happy, sad)               0.1097               Slightly related
(dog, cat)                 0.0809               Unrelated
(dog, algorithm)           0.0550               Unrelated


### Why Embeddings Matter for Prompting

The embedding space is the **first transformation** your prompt undergoes. When you write "Summarize this text:", the model doesn't see those words — it sees a specific trajectory through embedding space.

Key implications:
- **Synonyms land near each other** → "Summarize" and "Condense" activate similar model pathways
- **Domain terms cluster** → technical vocabulary activates different regions than casual language
- **The embedding quality limits the model** → if two concepts have similar embeddings, the model may confuse them

> **This is why word choice in prompts matters** — not because of mystical "prompt magic," but because different words literally navigate to different regions of the model's learned representation space.

---

## Part 4: The Attention Mechanism — How Context Shapes Predictions

### The Problem Attention Solves

Consider: "The animal didn't cross the road because **it** was too tired."

What does "it" refer to? The animal — because context tells us being "tired" applies to the animal, not the road. **The attention mechanism is how the model makes this connection.**

### How Attention Works (Simplified)

For each position in the sequence, the model asks: **"How relevant is every other position to predicting what comes next here?"**

```
Processing position: [it]
Attention weights:
  "The"      → 0.02  (low — generic article)
  "animal"   → 0.45  (HIGH — likely referent of "it")
  "didn't"   → 0.05  (low)
  "cross"    → 0.08  (medium — verb context)
  "the"      → 0.01  (low)
  "road"     → 0.15  (medium — alternative referent)
  "because"  → 0.12  (medium — explains relationship)
  "it"       → 0.07  (self-attention)
  "was"      → 0.03  (low)
  "too"      → 0.02  (low)
```

Qwen3-8B has **40 attention heads across 48 layers** — so it computes these relevance scores from 40 different "perspectives" at each of 48 levels of abstraction.

In [2]:
# === DEMONSTRATING ATTENTION: Same word, different context, different prediction ===

# The word "bank" means different things in different contexts
prompts = [
    "I need to deposit money at the bank",     # financial institution
    "The fisherman sat on the river bank",      # edge of water
    "The pilot performed a steep bank",         # aircraft maneuver
]

print("=== How Context Changes Prediction (Attention in Action) ===\n")

for prompt in prompts:
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(input_ids)

    # Get probabilities for what comes AFTER this prompt
    logits = outputs.logits[0, -1, :]
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, k=5)

    print(f"Prompt: '{prompt}'")
    print(f"  Top 5 next tokens:")
    for p, tid in zip(top_probs, top_ids):
        print(f"    '{tokenizer.decode([tid.item()])}' ({p.item():.4f})")
    print()

=== How Context Changes Prediction (Attention in Action) ===

Prompt: 'I need to deposit money at the bank'
  Top 5 next tokens:
    '.' (0.4199)
    ',' (0.2559)
    ' and' (0.1064)
    ' for' (0.0569)
    ' to' (0.0305)

Prompt: 'The fisherman sat on the river bank'
  Top 5 next tokens:
    ',' (0.8828)
    ' with' (0.0388)
    ' and' (0.0208)
    '.' (0.0183)
    ' one' (0.0142)

Prompt: 'The pilot performed a steep bank'
  Top 5 next tokens:
    ' and' (0.1787)
    ',' (0.1582)
    ' while' (0.1396)
    ' during' (0.1396)
    'ed' (0.1084)



### Why This Matters for Prompt Engineering

The attention mechanism explains several prompt engineering best practices:

1. **Put important context first (or last)** — attention has slight biases toward beginning and end of sequences ("primacy" and "recency" effects)
2. **System messages work because they're always attended to** — they're at the start of every sequence, so every subsequent token's prediction is influenced by them
3. **Long contexts dilute attention** — with more tokens, attention is spread thinner, which is why concise prompts often outperform verbose ones
4. **Delimiters help** — clear separators (like XML tags or markdown headers) create natural "attention boundaries" that help the model isolate different parts of your prompt

> **The attention mechanism is why prompt STRUCTURE matters, not just content.** The same information arranged differently produces different attention patterns and thus different outputs.

In [4]:
# === POSITIONAL EFFECTS: Order matters because of attention ===

# Same information, different order → different outputs
prompt_a = "Context: The patient has a fever of 103°F and severe headache.\nQuestion: What could be the diagnosis?"
prompt_b = "Question: What could be the diagnosis?\nContext: The patient has a fever of 103°F and severe headache."

messages_a = [{"role": "user", "content": prompt_a}]
messages_b = [{"role": "user", "content": prompt_b}]

print("=== Order Effects: Context-First vs Question-First ===\n")
print(f"Version A (context first):\n{generate(messages_a, max_new_tokens=150)}\n")
print(f"{'='*60}\n")
print(f"Version B (question first):\n{generate(messages_b, max_new_tokens=150)}")
print("\n💡 Notice: The model may emphasize different aspects based on order.")

=== Order Effects: Context-First vs Question-First ===

Version A (context first):
Based on the symptoms of **fever of 103°F (39.4°C)** and **severe headache**, several possible diagnoses could be considered. The most likely possibilities include:

### 1. **Viral Infection (e.g., Influenza, Common Cold, or Viral Meningitis)**
- Often presents with fever, headache, fatigue, and sometimes body aches.
- Viral meningitis can cause severe headache and high fever, along with neck stiffness.

### 2. **Bacterial Infection (e.g., Bacterial Meningitis, Sinusitis, or Tonsillitis)**
- Bacterial meningitis is a medical emergency and can cause severe headache, high fever


Version B (question first):
The diagnosis could be **meningitis** or **encephalitis**, especially given the presence of a **high fever (103°F)** and **severe headache**. These symptoms are classic for central nervous system infections.

### Possible Diagnoses:
1. **Bacterial Meningitis**
   - Most severe form of meningitis.
   - P

---

## Part 5: Generation Algorithms — From Probabilities to Text

### The Decision Point

After the model computes probabilities for the next token, we need a **strategy** to pick one. This choice fundamentally affects output quality, creativity, and consistency.

### Temperature: The Master Control

Temperature scales the logits BEFORE softmax. Mathematically:

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

Where:
- $z_i$ = raw logit for token $i$
- $T$ = temperature
- $p_i$ = probability after temperature scaling

**Low temperature (0.1-0.3)**: Sharpens the distribution → the top token gets almost all probability → near-deterministic output  
**Temperature = 1.0**: No change — uses the model's natural distribution  
**High temperature (1.5+)**: Flattens the distribution → more uniform probabilities → more random/creative output

In [9]:
# === TEMPERATURE VISUALIZATION: How it reshapes the probability distribution ===

prompt = "Carbon is a very"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(input_ids)

logits = outputs.logits[0, -1, :].float()  # Raw logits

temperatures = [0.1, 0.3, 0.7, 1.0, 1.5, 2.0]

print(f"Prompt: '{prompt}'")
print(f"\n{'Temp':<6} {'Top-1 prob':<12} {'Top-5 prob':<12} {'Top-10 prob':<12} {'Top token':<15} {'Distribution'}")
print("-" * 80)

for T in temperatures:
    scaled_logits = logits / T
    probs = torch.softmax(scaled_logits, dim=-1)

    top_probs, top_ids = torch.topk(probs, k=10)
    top1_prob = top_probs[0].item()
    top5_prob = top_probs[:5].sum().item()
    top10_prob = top_probs[:10].sum().item()
    top_token = tokenizer.decode([top_ids[0].item()])

    # Visual bar showing concentration
    bar = "█" * int(top1_prob * 30) + "░" * (30 - int(top1_prob * 30))
    print(f"{T:<6.1f} {top1_prob:<12.4f} {top5_prob:<12.4f} {top10_prob:<12.4f} '{top_token}'{'':>{12-len(top_token)}} {bar}")

print("\n💡 At T=0.1, the top token dominates (~deterministic). At T=2.0, probability is spread widely (~random).")

Prompt: 'Carbon is a very'

Temp   Top-1 prob   Top-5 prob   Top-10 prob  Top token       Distribution
--------------------------------------------------------------------------------
0.1    1.0000       1.0000       1.0000       ' important'   ██████████████████████████████
0.3    0.9999       1.0000       1.0000       ' important'   █████████████████████████████░
0.7    0.9666       0.9954       0.9991       ' important'   ████████████████████████████░░
1.0    0.8467       0.9506       0.9770       ' important'   █████████████████████████░░░░░
1.5    0.4429       0.6113       0.6847       ' important'   █████████████░░░░░░░░░░░░░░░░░
2.0    0.1094       0.1835       0.2257       ' important'   ███░░░░░░░░░░░░░░░░░░░░░░░░░░░

💡 At T=0.1, the top token dominates (~deterministic). At T=2.0, probability is spread widely (~random).


### Sampling Strategies

Once we have probabilities (possibly temperature-scaled), we need a sampling strategy:

| Strategy | How it works | When to use |
|----------|-------------|-------------|
| **Greedy** | Always pick the highest probability token | Factual tasks, deterministic output needed |
| **Random sampling** | Sample from the full distribution weighted by probability | Maximum creativity, high variance |
| **Top-k** | Only consider the top k tokens, redistribute probability among them | Balanced creativity with some constraint |
| **Top-p (nucleus)** | Only consider tokens until cumulative probability reaches p | Adaptive — uses fewer tokens when model is confident |

Let's see each in action:

In [10]:
# === GENERATION STRATEGIES COMPARED ===

prompt = "Once upon a time in a land far away, there lived a"

print(f"Prompt: '{prompt}'\n")
print("=" * 70)

# Strategy 1: Greedy (do_sample=False)
messages = [{"role": "user", "content": f"Continue this story in one sentence: {prompt}"}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

print("\n1. GREEDY (always pick highest probability):")
output = model.generate(**inputs, max_new_tokens=50, do_sample=False)
generated = output[0][inputs.input_ids.shape[1]:]
print(f"   {tokenizer.decode(generated, skip_special_tokens=True)}")

# Strategy 2: Sampling with temperature
print("\n2. SAMPLING (T=1.0, sample from full distribution):")
for i in range(3):
    output = model.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=1.0, top_k=0)
    generated = output[0][inputs.input_ids.shape[1]:]
    print(f"   Run {i+1}: {tokenizer.decode(generated, skip_special_tokens=True)}")

# Strategy 3: Top-k sampling
print("\n3. TOP-K (k=20, only consider top 20 tokens):")
for i in range(3):
    output = model.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=0.7, top_k=20)
    generated = output[0][inputs.input_ids.shape[1]:]
    print(f"   Run {i+1}: {tokenizer.decode(generated, skip_special_tokens=True)}")

# Strategy 4: Top-p (nucleus) sampling
print("\n4. TOP-P / NUCLEUS (p=0.9, adaptive token set):")
for i in range(3):
    output = model.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=0.7, top_p=0.9, top_k=0)
    generated = output[0][inputs.input_ids.shape[1]:]
    print(f"   Run {i+1}: {tokenizer.decode(generated, skip_special_tokens=True)}")

print("\n💡 Greedy: always the same. Sampling: varied. Top-k/top-p: varied but constrained.")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Prompt: 'Once upon a time in a land far away, there lived a'


1. GREEDY (always pick highest probability):
   Once upon a time in a land far away, there lived a young girl with the power to speak to the stars, but she was forbidden to use it by the ancient council that ruled the sky.

2. SAMPLING (T=1.0, sample from full distribution):
   Run 1: Once upon a time in a land far away, there lived a young sorceress with the power to weave dreams into reality, but she was cursed to only dream of the world she longed to save.
   Run 2: Once upon a time in a land far away, there lived a brave young knight who had a secret—the power to speak to the stars.
   Run 3: Once upon a time in a land far away, there lived a curious little fox who could talk to the stars and dream of adventures beyond the horizon.

3. TOP-K (k=20, only consider top 20 tokens):
   Run 1: Once upon a time in a land far away, there lived a young girl with a compass that pointed not north, but toward the hearts of those sh

In [11]:
# === TOP-K vs TOP-P: Understanding the difference ===

prompt = "The best programming language for beginners is"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(input_ids)

logits = outputs.logits[0, -1, :].float()
probs = torch.softmax(logits / 0.7, dim=-1)  # Temperature 0.7
sorted_probs, sorted_ids = torch.sort(probs, descending=True)

print(f"Prompt: '{prompt}'\n")

# Show what top-k=20 selects
print("TOP-K (k=20): Always selects exactly 20 tokens")
print(f"  Tokens included: 20")
print(f"  Combined probability: {sorted_probs[:20].sum().item():.4f}")
print(f"  Top 5: {[tokenizer.decode([sorted_ids[i].item()]) for i in range(5)]}")

# Show what top-p=0.9 selects
cumsum = torch.cumsum(sorted_probs, dim=-1)
p_cutoff = (cumsum <= 0.9).sum().item() + 1
print(f"\nTOP-P (p=0.9): Adaptively selects tokens until 90% cumulative probability")
print(f"  Tokens included: {p_cutoff}")
print(f"  Combined probability: {sorted_probs[:p_cutoff].sum().item():.4f}")
print(f"  Top 5: {[tokenizer.decode([sorted_ids[i].item()]) for i in range(5)]}")

print(f"\n💡 Top-k is fixed (always 20 tokens).")
print(f"   Top-p is adaptive ({p_cutoff} tokens here, but could be more or fewer for other prompts).")
print(f"   When the model is confident, top-p uses fewer tokens. When uncertain, it uses more.")

Prompt: 'The best programming language for beginners is'

TOP-K (k=20): Always selects exactly 20 tokens
  Tokens included: 20
  Combined probability: 0.9864
  Top 5: [' a', ' the', ' Python', ' one', ' not']

TOP-P (p=0.9): Adaptively selects tokens until 90% cumulative probability
  Tokens included: 5
  Combined probability: 0.9101
  Top 5: [' a', ' the', ' Python', ' one', ' not']

💡 Top-k is fixed (always 20 tokens).
   Top-p is adaptive (5 tokens here, but could be more or fewer for other prompts).
   When the model is confident, top-p uses fewer tokens. When uncertain, it uses more.


### Choosing Generation Parameters: A Practical Guide

| Task | Temperature | Strategy | Why |
|------|------------|----------|-----|
| **Factual Q&A** | 0.1-0.3 | Greedy or top-p=0.9 | Want consistent, accurate answers |
| **Code generation** | 0.2-0.5 | Top-p=0.95 | Need correctness with some flexibility |
| **Creative writing** | 0.7-1.0 | Top-p=0.9, top-k=40 | Want variety and creativity |
| **Brainstorming** | 0.9-1.2 | Top-k=50 | Want diverse, unexpected ideas |
| **Classification** | 0.0 | Greedy | Want deterministic labels |

> **Rule of thumb**: Start with temperature=0.7, top_k=20. Decrease temperature for factual tasks, increase for creative tasks.

---

## Part 6: Chat Templates — The Hidden Formatting Layer

### Why Chat Templates Exist

When you write `[{"role": "user", "content": "Hello"}]`, the model doesn't see a JSON object. The **chat template** converts your message list into a specific text format with **special control tokens** that the model learned to recognize during instruction tuning.

Different models use different formats:
- **ChatML** (used by Qwen): `<|im_start|>user\nHello<|im_end|>`
- **Llama format**: `[INST] Hello [/INST]`
- **Alpaca format**: `### Instruction:\nHello\n### Response:\n`

Using the wrong template → the model sees garbage → poor responses. This is why `apply_chat_template()` exists — it handles the formatting automatically.

In [12]:
# === REVEALING THE CHAT TEMPLATE: What the model actually sees ===

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
]

# Show the raw formatted text (what gets tokenized)
raw_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

print("=== Messages you write ===")
for msg in messages:
    print(f"  [{msg['role']}]: {msg['content']}")

print(f"\n=== Raw text the model sees (after template formatting) ===")
print(raw_text)

print(f"\n=== Character count: {len(raw_text)} ===")

=== Messages you write ===
  [system]: You are a helpful assistant.
  [user]: What is 2+2?

=== Raw text the model sees (after template formatting) ===
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is 2+2?<|im_end|>
<|im_start|>assistant
<think>

</think>



=== Character count: 139 ===


In [13]:
# === TOKEN-LEVEL VIEW: Every token the model processes ===

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
]

# Tokenize WITH special tokens visible
raw_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
token_ids = tokenizer.encode(raw_text, add_special_tokens=False)

print(f"Total tokens in this conversation: {len(token_ids)}\n")
print(f"{'Position':<10} {'Token ID':<10} {'Token text':<25} {'Type'}")
print("-" * 60)

# Identify special vs regular tokens
special_token_ids = set(tokenizer.all_special_ids)
for pos, tid in enumerate(token_ids):
    token_text = tokenizer.decode([tid])
    raw_token = tokenizer.convert_ids_to_tokens([tid])[0]
    is_special = tid in special_token_ids or raw_token.startswith("<|")
    token_type = "🔵 CONTROL" if is_special else "  regular"
    # Escape newlines for display
    display_text = token_text.replace("\n", "⏎")
    print(f"{pos:<10} {tid:<10} '{display_text}'{'':>{22-len(display_text)}} {token_type}")

Total tokens in this conversation: 30

Position   Token ID   Token text                Type
------------------------------------------------------------
0          151644     '<|im_start|>'           🔵 CONTROL
1          8948       'system'                   regular
2          198        '⏎'                        regular
3          2610       'You'                      regular
4          525        ' are'                     regular
5          264        ' a'                       regular
6          10950      ' helpful'                 regular
7          17847      ' assistant'               regular
8          13         '.'                        regular
9          151645     '<|im_end|>'             🔵 CONTROL
10         198        '⏎'                        regular
11         151644     '<|im_start|>'           🔵 CONTROL
12         872        'user'                     regular
13         198        '⏎'                        regular
14         3838       'What'                     

### What the Template Reveals

Look at the output above carefully:

1. **Control tokens** (🔵) are **special markers** that don't appear in normal text. The model learned during training that these mark role boundaries.
2. **The system message** is just regular text wrapped in control tokens — it has no magical status, just a learned convention.
3. **`add_generation_prompt=True`** adds the assistant's opening marker, telling the model "start generating the assistant's response now."
4. **Every character costs tokens** — even the role labels and newlines use tokens from your context budget.

### Why `enable_thinking=False` vs `True`

Qwen3 supports a "thinking mode" where it generates internal reasoning in a `<think>` block before the actual answer. This is controlled by a special token in the template:

In [14]:
# === THINKING MODE: How the template changes ===

messages = [{"role": "user", "content": "What is 15 × 37?"}]

# Without thinking
text_no_think = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
# With thinking
text_think = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)

print("=== Without thinking mode ===")
print(text_no_think)
print(f"\n=== With thinking mode ===")
print(text_think)
print(f"\n💡 The only difference is a special token that tells the model")
print(f"   whether to show its reasoning process or jump straight to the answer.")

=== Without thinking mode ===
<|im_start|>user
What is 15 × 37?<|im_end|>
<|im_start|>assistant
<think>

</think>



=== With thinking mode ===
<|im_start|>user
What is 15 × 37?<|im_end|>
<|im_start|>assistant


💡 The only difference is a special token that tells the model
   whether to show its reasoning process or jump straight to the answer.


---

## Part 7: Quantization — Fitting a Giant Model on a Small GPU

### The Memory Problem

Qwen3-8B has **14.8 billion parameters**. In full precision (FP32 = 32 bits = 4 bytes per parameter):

```
14.8B × 4 bytes = 59.2 GB  ← Doesn't fit on any consumer GPU!
```

In half precision (FP16/BF16 = 16 bits = 2 bytes):
```
14.8B × 2 bytes = 29.6 GB  ← Still too big for T4 (16GB) or even RTX 4090 (24GB)
```

### The Solution: 4-bit Quantization (NF4)

We use **NF4 (Normal Float 4-bit)** quantization, which represents each parameter with just 4 bits:

```
14.8B × 0.5 bytes = 7.4 GB  ← Fits on a T4 with room for activations!
```

### How NF4 Works

Regular 4-bit would divide the range `[-max, +max]` into 16 equally spaced values. But neural network weights follow a **normal (Gaussian) distribution** — most values are near zero, few are at the extremes.

**NF4 is smart**: it places the 16 quantization levels at the **quantiles** of the normal distribution, so more precision is allocated where most weights actually are. This reduces information loss compared to naive 4-bit.

In [2]:
# === QUANTIZATION IN PRACTICE: Memory and quality ===

print("=== Model Memory Analysis ===\n")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"\nTheoretical memory at different precisions:")
print(f"  FP32 (32-bit): {total_params * 4 / 1e9:.1f} GB")
print(f"  FP16 (16-bit): {total_params * 2 / 1e9:.1f} GB")
print(f"  INT8 (8-bit):  {total_params * 1 / 1e9:.1f} GB")
print(f"  NF4 (4-bit):   {total_params * 0.5 / 1e9:.1f} GB")
print(f"\nActual GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print(f"GPU memory reserved:    {torch.cuda.memory_reserved() / 1e9:.1f} GB")
print(f"GPU total memory:       {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Show quantization config
print(f"\n=== Active Quantization Config ===")
print(f"  Method: 4-bit NF4")
print(f"  Compute dtype: bfloat16 (computations upcast to 16-bit for accuracy)")
print(f"  Quant type: nf4 (Normal Float — optimized for weight distributions)")

# Count quantized vs non-quantized parameters
quantized = 0
not_quantized = 0
for name, param in model.named_parameters():
    if hasattr(param, 'quant_state') or '4bit' in str(param.dtype).lower():
        quantized += param.numel()
    else:
        not_quantized += param.numel()

print(f"\n💡 The model stores weights in 4 bits but computes in 16 bits.")
print(f"   This gives us ~95% of full-precision quality at 1/8 the memory.")

=== Model Memory Analysis ===

Total parameters: 4,717,851,648

Theoretical memory at different precisions:
  FP32 (32-bit): 18.9 GB
  FP16 (16-bit): 9.4 GB
  INT8 (8-bit):  4.7 GB
  NF4 (4-bit):   2.4 GB

Actual GPU memory used: 6.4 GB
GPU memory reserved:    15.4 GB


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

### The Quality Trade-off

4-bit quantization is not free — there is a small quality loss:

| Precision | Memory | Quality (approximate) | Use case |
|-----------|--------|----------------------|----------|
| FP32 | 59 GB | 100% (baseline) | Research, exact evaluation |
| FP16/BF16 | 30 GB | ~99.9% | Training, high-end inference |
| INT8 | 15 GB | ~99% | Production serving |
| **NF4** | **7.4 GB** | **~95-97%** | **Consumer GPUs, Colab** |

For **learning and prompt engineering**, the difference is negligible. The patterns and techniques you learn here work identically with full-precision models.

> **Key insight**: Quantization is a **deployment optimization**, not a fundamental limitation. Everything in this course applies to any model at any precision.

---

## Part 8: Putting It All Together — The Complete Pipeline

Now you understand every step from your text to the model's response. Let's trace the full pipeline:

In [3]:
# === THE COMPLETE PIPELINE: From text to response, every step visible ===

user_message = "Explain gravity in one sentence."
messages = [
    {"role": "system", "content": "You are a physics teacher. Be concise."},
    {"role": "user", "content": user_message},
]

print("=" * 70)
print("STEP 1: Your messages")
print("=" * 70)
for m in messages:
    print(f"  [{m['role']}] {m['content']}")

print(f"\n{'=' * 70}")
print("STEP 2: Chat template → formatted text")
print("=" * 70)
formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
print(formatted)

print(f"\n{'=' * 70}")
print("STEP 3: Tokenization → token IDs")
print("=" * 70)
input_ids = tokenizer.encode(formatted, return_tensors="pt").to(model.device)
print(f"  {input_ids.shape[1]} tokens: {input_ids[0].tolist()[:20]}... (showing first 20)")

print(f"\n{'=' * 70}")
print("STEP 4: Embedding → vectors")
print("=" * 70)
with torch.no_grad():
    embeddings = model.model.embed_tokens(input_ids)
print(f"  Embedding shape: {embeddings.shape}")
print(f"  → {embeddings.shape[1]} tokens × {embeddings.shape[2]} dimensions")

print(f"\n{'=' * 70}")
print("STEP 5: Transformer layers (attention + feed-forward) × 48")
print("=" * 70)
print(f"  Model has {model.config.num_hidden_layers} transformer layers")
print(f"  Each layer has {model.config.num_attention_heads} attention heads")
print(f"  Processing...")

with torch.no_grad():
    outputs = model(input_ids)

print(f"  Output logits shape: {outputs.logits.shape}")
print(f"  → Probability distribution over {outputs.logits.shape[2]:,} tokens")

print(f"\n{'=' * 70}")
print("STEP 6: Logits → softmax → probabilities → sampling → tokens")
print("=" * 70)

logits = outputs.logits[0, -1, :].float()
probs = torch.softmax(logits / 0.7, dim=-1)  # Temperature 0.7
top_probs, top_ids = torch.topk(probs, k=5)

print(f"  Top 5 next-token candidates:")
for p, tid in zip(top_probs, top_ids):
    print(f"    '{tokenizer.decode([tid.item()])}' → {p.item():.4f}")

print(f"\n{'=' * 70}")
print("STEP 7: Autoregressive loop → complete response")
print("=" * 70)
response = generate(messages, max_new_tokens=100)
print(f"  {response}")
print(f"\n✅ Pipeline complete: text → tokens → embeddings → attention → logits → sampling → text")

STEP 1: Your messages
  [system] You are a physics teacher. Be concise.
  [user] Explain gravity in one sentence.

STEP 2: Chat template → formatted text
<|im_start|>system
You are a physics teacher. Be concise.<|im_end|>
<|im_start|>user
Explain gravity in one sentence.<|im_end|>
<|im_start|>assistant
<think>

</think>



STEP 3: Tokenization → token IDs
  33 tokens: [151644, 8948, 198, 2610, 525, 264, 21321, 11079, 13, 2823, 63594, 13, 151645, 198, 151644, 872, 198, 840, 20772, 23249]... (showing first 20)

STEP 4: Embedding → vectors
  Embedding shape: torch.Size([1, 33, 4096])
  → 33 tokens × 4096 dimensions

STEP 5: Transformer layers (attention + feed-forward) × 48
  Model has 36 transformer layers
  Each layer has 32 attention heads
  Processing...
  Output logits shape: torch.Size([1, 33, 151936])
  → Probability distribution over 151,936 tokens

STEP 6: Logits → softmax → probabilities → sampling → tokens
  Top 5 next-token candidates:
    'Gravity' → 1.0000
    ' Gravity' → 0

---

## Part 9: Your First Prompt Engineering (With Understanding)

Now that you understand the internals, let's see how prompt engineering techniques connect to what's happening inside the model.

In [4]:
# === EXPERIMENT 1: Vague vs Specific prompts ===
# WHY this works: Specific words activate more focused attention patterns,
# narrowing the probability distribution toward relevant tokens.

vague = "Tell me about AI."
specific = "Explain three ways transformer-based language models are used in healthcare, with one concrete example each."

print("=== Vague Prompt ===")
print(f"Prompt: {vague}")
print(f"Response: {generate([{'role': 'user', 'content': vague}], max_new_tokens=200)}")
print()
print("=== Specific Prompt ===")
print(f"Prompt: {specific}")
print(f"Response: {generate([{'role': 'user', 'content': specific}], max_new_tokens=200)}")
print()
print("💡 The specific prompt narrows the model's attention to relevant knowledge regions,")
print("   producing focused, structured output instead of a broad overview.")

=== Vague Prompt ===
Prompt: Tell me about AI.
Response: Artificial Intelligence, or AI, refers to the simulation of human intelligence in machines that are programmed to think, learn, and perform tasks that typically require human intelligence. The concept of AI has evolved significantly over the years, and today it encompasses a wide range of technologies and applications.

### Key Characteristics of AI:
- **Learning**: AI systems can learn from data, improving their performance over time without being explicitly programmed for each task.
- **Reasoning**: AI can solve complex problems by using logic and reasoning.
- **Problem Solving**: AI can find solutions to problems that may involve multiple steps or variables.
- **Understanding Language**: Many AI systems can understand and generate human language.
- **Perception**: AI can process and interpret sensory information, such as images, sounds, and text.

### Types of AI:
1. **Narrow AI (or Weak AI)**: These are AI systems designed fo

In [5]:
# === EXPERIMENT 2: System messages shape the entire output ===
# WHY: System tokens are at the START of the sequence, so every subsequent
# attention computation includes them. They're a persistent bias on all output.

user_msg = "What is machine learning?"

personas = [
    ("Default (no system message)", None),
    ("5-year-old teacher", "You explain everything as if talking to a 5-year-old. Use simple words and fun analogies."),
    ("PhD professor", "You are a computer science professor. Use precise technical terminology and cite foundational concepts."),
    ("Pirate", "You are a pirate. You speak entirely in pirate dialect and use nautical metaphors."),
]

for name, system_msg in personas:
    messages = []
    if system_msg:
        messages.append({"role": "system", "content": system_msg})
    messages.append({"role": "user", "content": user_msg})

    print(f"\n=== {name} ===")
    print(generate(messages, max_new_tokens=150))
    print("-" * 60)


=== Default (no system message) ===
Machine learning is a subset of **artificial intelligence (AI)** that focuses on building systems that **learn from data**, identify patterns, and make decisions with minimal human intervention. The goal of machine learning is to enable computers to improve at a task over time through experience (i.e., data), without being explicitly programmed for each specific task.

---

### 🔍 **Key Characteristics of Machine Learning:**

1. **Learning from Data**:
   - Machine learning models are trained on large datasets.
   - They learn patterns and relationships within the data.

2. **Adaptive and Improving**:
   - As more data is provided, the model can improve its performance.

3. **No Explicit Programming**:
   - Instead of being programmed with
------------------------------------------------------------

=== 5-year-old teacher ===
Hey there! Let me explain **machine learning** in a super fun way—like a story!

Imagine you have a **robot friend** who want

In [6]:
# === EXPERIMENT 3: Temperature's practical effect ===
# WHY: Temperature scales logits before softmax. Low T → peaked distribution
# (predictable), high T → flat distribution (creative/random).

prompt = "Invent a name for a new color that doesn't exist."
messages = [{"role": "user", "content": prompt}]

print(f"Prompt: '{prompt}'\n")

for temp in [0.1, 0.5, 0.7, 1.0, 1.5]:
    print(f"Temperature {temp}:")
    for i in range(3):
        result = generate(messages, max_new_tokens=30, temperature=temp)
        # Take just the first line
        first_line = result.strip().split("\n")[0]
        print(f"  Run {i+1}: {first_line}")
    print()

Prompt: 'Invent a name for a new color that doesn't exist.'

Temperature 0.1:
  Run 1: How about **"Lunaria"**?
  Run 2: Sure! How about **"Lunara"**?
  Run 3: How about **"Lunaria"**?

Temperature 0.5:
  Run 1: How about **"Lunaria"**?
  Run 2: Sure! How about **"Lunara"**?
  Run 3: Sure! How about **"Lunara"**?

Temperature 0.7:
  Run 1: How about **"Lunaris"**?
  Run 2: How about **"Lunara"**?
  Run 3: How about **"Lunaria"**?

Temperature 1.0:
  Run 1: Sure! How about **"Lunaria"**?
  Run 2: Sure! Here's a name for a new, imaginary color:
  Run 3: Certainly! How about **"Zephyron"**?

Temperature 1.5:
  Run 1: Sure! How about **"Zephyrion"**?
  Run 2: Sure! How about **"Zephyrine"?**
  Run 3: Certainly! How about **"Sapphirethunder"**?



## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** rewrite a prompt using the technique from this notebook. Document what changes and why.

**Exercise 2 — Build:** test the technique on a different task and compare results. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** combine this technique with one from a previous notebook.

---

## Summary: The Mental Model

You now have a complete mental model of how LLMs work:

```
Your prompt (text)
    ↓ Tokenizer (BPE)
Token IDs (integers)
    ↓ Embedding layer
Vectors (high-dimensional)
    ↓ Transformer layers × 48 (attention + feed-forward)
Contextualized representations
    ↓ Output head
Logits (one score per vocabulary token)
    ↓ Temperature scaling
Scaled logits
    ↓ Softmax
Probability distribution
    ↓ Sampling strategy (greedy / top-k / top-p)
Next token ID
    ↓ Tokenizer (decode)
Generated text (one token)
    ↓ Repeat (autoregressive loop)
Complete response
```

### Key Takeaways for Prompt Engineering

1. **Prompts are just input tokens** that shape the probability distribution over output tokens
2. **Word choice matters** because different words → different embeddings → different attention → different predictions
3. **Structure matters** because delimiters and formatting affect attention patterns
4. **System messages work** because they're attended to by every subsequent token
5. **Temperature and sampling** are your knobs for controlling the creativity-consistency trade-off
6. **Token count ≠ word count** — always think in tokens for context limits

### What's Next

The remaining notebooks in this series build on these foundations:
- **Notebook 2**: Prompt structures (single-turn, multi-turn, chat templates in practice)
- **Notebook 3**: Template engineering (Jinja2, dynamic prompts)
- **Notebooks 4-6**: Zero-shot, few-shot, and chain-of-thought techniques
- **Notebooks 7-9**: Self-consistency, constrained generation, role prompting
- **Notebooks 10+**: Advanced patterns (decomposition, chaining, optimization, security)

Every technique will connect back to the concepts you learned here. When something "just works," you'll understand *why*.

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the prompt-engineering/ directory